In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

## What is RAG? 

Retrieval Augmented Generation

## What does that mean? 
1. We search a knowledge base (retrieval) to get relevant information to a user's question. 
2. We use that information to generate a response based on the user's question and the retrieved information. 
3. We return the generated response to the user. 

## Step 0: Create the knowledge base

In [3]:
import requests

In [4]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
courses_raw

[{'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 26},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41}]

In [6]:
documents = []

url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data) 

len(documents)



1275

In [7]:
documents[900]

{'id': 'e5c67e3439',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 10. Kubernetes and TensorFlow Serving',
 'question': 'Kubernetes-dashboard',
 'answer': '[Deploy and Access the Kubernetes Dashboard](https://kubernetes.io/docs/tasks/access-application-cluster/web-ui-dashboard/)'}

## Step 1: Create the Retriever (aka, search our knowledge base)

But first, why not just let the LLM search the entire knowledge base and generate a response? 

Well, we could do that, but it would be incredibly inefficient and expensive. LLMs utilize a context window ,(i.e., the amount of text it can process at any one time) so, if we pass a tremendous amount of data to the LLM, it will have to look over everthing we send it. The more we send, the more it costs, the longer it takes, and the more likely it is to make a mistake by being overwhelmed with information; think cows in snowstorm. 

So, let's limit what it has to sift through by searching our knowledge base BEFORE sending it to the LLM. 

In [8]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [9]:
question = "I just discovered the course. Can I still join?"

question

'I just discovered the course. Can I still join?'

In [10]:
search_results = index.search(question, 
             filter_dict={'course':'llm-zoomcamp'},
             num_results=5
             )

In [11]:
def search(question, course='llm-zoomcamp'): 
    search_results = index.search(question, 
            boost_dict={'question': 2, 'section': 0.5,}, 
             filter_dict={'course':course},
             num_results=5
    ) 

    return search_results

search_results = search(question)

## Step 2: Build the Prompt based on the question and the retrieved information

Prompts are decomposed into two parts: the system prompt and the user prompt. 

### System prompt 

Instructions/limitations for the LLM. 

### User prompt
The question asked by the user. 

In [12]:
context = '''

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don't post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as "Lucid Elbakyan." Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
This course is being offered for the first time, and things will keep changing until a given module is ready, at which point it shall be announced. Working on the material or homework in advance will be at your own risk, as the final version could be different.
'''

In [13]:
INSTRUCTIONS = f'''
Your task it to answer questions based on the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."
'''

USER_PROMPT_TEMPLATE = f''' 
Question: 
{question}

Context: 
{context}
''' 

In [14]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q:' + doc['question'])
        lines.append('A:' + doc['answer'])
        lines.append('')
    
    return '\n'.join(lines).strip()

In [15]:
def build_prompt(question, search_results): 
    context = build_context(search_results)
    prompt =  USER_PROMPT_TEMPLATE.format(question=question, context=context)

    return prompt.strip()



In [16]:
prompt = build_prompt(question, search_results)

print(prompt)

Question: 
I just discovered the course. Can I still join?

Context: 


I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can als

## Step 3: Return the Generated Response to the User

In [17]:
response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )

In [18]:
# We can calculate the cost of the response by investigating the usage infrormation from the API
print(response.usage)

ResponseUsage(input_tokens=780, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=40, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=820)


In [19]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0007650000000000001

In [20]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS },
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history
    )

In [21]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
            model=model,
            input=message_history
        )
    
    return response.output_text

In [22]:
# Putting it all together 

def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [23]:
answer = rag(question)

print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.
